# Hyperparameter Tuning
The purpose of this notebook is to improve our baseline ML models, tuning their hyperparameters and selecting the strongest model based on evidence. Both the Logistic Regression and Random Forest models will be tuned and then the best model will be saved for future notebooks.

In [0]:
import joblib
import numpy as np
from pathlib import Path
from scipy.sparse import issparse

# Get this notebook’s Workspace path, then jump to repo root and into /artifacts
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()                    # like: /Users/.../csai382.../notebooks/csai_lab_5_3
repo_ws_path = nb_ws_path.rsplit("/notebooks/", 1)[0]   # like: /Users/.../csai382...
ART = Path("/Workspace" + repo_ws_path) / "artifacts"

# Load artifacts (match your screenshot)
pipeline = joblib.load(ART / "stedi_feature_pipeline.pkl")

X_train_transformed = np.load(ART / "X_train_transformed.npy", allow_pickle=True)
X_test_transformed  = np.load(ART / "X_test_transformed.npy", allow_pickle=True)

y_train = joblib.load(ART / "y_train.pkl")
y_test  = joblib.load(ART / "y_test.pkl")

def to_float_matrix(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 0:
        arr = arr.item()
        if issparse(arr):
            arr = arr.toarray()
        arr = np.array(arr, dtype=float)
    elif arr.dtype == object:
        arr = np.array([
            x.toarray() if issparse(x) else np.array(x, dtype=float)
            for x in arr
        ])
        arr = np.vstack(arr)
    elif issparse(arr):
        arr = arr.toarray()
    else:
        arr = np.array(arr, dtype=float)
    return arr

X_train = to_float_matrix(X_train_transformed)
X_test  = to_float_matrix(X_test_transformed)

y_train = np.ravel(y_train)
y_test  = np.ravel(y_test)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

log_reg_params = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

log_reg_grid = GridSearchCV(
    LogisticRegression(max_iter=300),
    log_reg_params,
    cv=3,
    scoring="accuracy"
)

log_reg_grid.fit(X_train, y_train)

log_reg_best_params = log_reg_grid.best_params_
log_reg_best_score = log_reg_grid.best_score_

log_reg_best_params, log_reg_best_score


# Logistic Regression Parameters

Best parameters: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}

Best score: 0.9511214840660257

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(),
    rf_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best_params = rf_grid.best_params_
rf_best_score = rf_grid.best_score_

rf_best_params, rf_best_score


# Random Forest Parameters
Best Parameters: {'max_depth': 5,
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 50}

Best Score: 0.9511214840660257

In [0]:
results = {
    "Logistic Regression (tuned)": log_reg_best_score,
    "Random Forest (tuned)": rf_best_score
}
results

Both Logistic Regression and Random Forest have identical scores upon tuning; due to this, and the fact that Random Forest takes 10+ times as long to run than Logistic Regression, without any real benefits, we are going to default to Logistic Regression for the additional speed that we gain.

In [0]:
# Choose the better model based on best_score_
if rf_best_score > log_reg_best_score:
    best_model = rf_grid.best_estimator_
    best_model_name = "Random Forest"
else:
    best_model = log_reg_grid.best_estimator_
    best_model_name = "Logistic Regression"

best_model_name, best_model

In [0]:
import joblib
from pathlib import Path

# Find this repo's /artifacts folder from the notebook path
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()
repo_ws_path = nb_ws_path.rsplit("/notebooks/", 1)[0]
ART = Path("/Workspace" + repo_ws_path) / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

# Save into repo artifacts
joblib.dump(best_model, str(ART / "stedi_best_model.pkl"))


# Model Evaluation and Ethics Reflection
According to our tests after tuning each of the models, we find that both result in identical test scores. Because of this, we default to which one performed faster, which was Logistic Regression. We scored specifically on accuracy, resulting in identical scores. In markdown cells above this you will find the best resulting parameters for accuracy for each model.

I was surprised to find that both models were just as accurate; I had assumed that the faster one sacrificed accuracy for speed, but it seems that is not the case. If I had more time, I'd be trying out a variety of models, increased data, random values in each of the parameters, even scoring based on other criteria instead of accuracy.

I scored based on accuracy, as mentioned before. One thing to be aware of is how scoring entirely on accuracy could result in some kind of bias. Hence, if I clearly state what I am scoring on, I exhibit proper transparency for what we are doing.

Repentance in the Gospel requires we look at what we have done from God's perspective, decide what we did was wrong and why, and then turn to God for help to stop doing such actions, thought patterns, etc. in an attempt to become better and more like Him.